# Tema 9 · Buenas prácticas: que tu cuaderno sobreviva
*Curso «Google Colab» · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** que un entrenamiento largo sobreviva a una desconexión, que tus resultados se puedan repetir y evitar
los errores más comunes en los cuadernos compartidos. Este tema no necesita GPU.

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
# La prueba automática de MiAtlas.cl define MIATLAS_PRUEBA=1 para correr con pocos datos. Tú no necesitas tocarla.
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"

import torch
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
print("Colab:", EN_COLAB, "| dispositivo:", DISPOSITIVO, "| prueba rápida:", PRUEBA_RAPIDA)
if DISPOSITIVO == "cpu":
    print("⚠ Sin GPU: el cuaderno funciona, pero más lento. En Colab: Entorno de ejecución → Cambiar tipo → GPU.")

In [ ]:
from pathlib import Path
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA = Path("/content/drive/MyDrive/MiAtlas-Colab")
else:
    CARPETA = Path("MiAtlas-Colab")
CARPETA.mkdir(parents=True, exist_ok=True)
print("Carpeta de trabajo:", CARPETA)

## 1. Puntos de control: retomar tras una desconexión

En el plan gratis la máquina puede apagarse (12 h como máximo, inactividad o falta de recursos, según la
[FAQ](https://research.google.com/colaboratory/faq.html)). Si entrenas por horas, **guarda un punto de control en Drive cada cierto tiempo** y, al volver,
continúa desde ahí.

In [ ]:
from torch import nn

PUNTO = CARPETA / "punto_de_control.pt"

def guardar_punto(epoca, modelo, optimizador):
    torch.save({"epoca": epoca, "modelo": modelo.state_dict(), "optimizador": optimizador.state_dict()}, PUNTO)

def cargar_punto(modelo, optimizador):
    """Devuelve la época desde donde seguir (0 si no hay punto de control)."""
    if not PUNTO.exists():
        return 0
    estado = torch.load(PUNTO, map_location=DISPOSITIVO)
    modelo.load_state_dict(estado["modelo"])
    optimizador.load_state_dict(estado["optimizador"])
    return estado["epoca"]

modelo = nn.Linear(10, 1).to(DISPOSITIVO)
optimizador = torch.optim.SGD(modelo.parameters(), lr=0.1)
desde = cargar_punto(modelo, optimizador)
print("Se continúa desde la época", desde)
for epoca in range(desde + 1, desde + 4):
    x = torch.randn(64, 10, device=DISPOSITIVO)
    perdida = (modelo(x) - x.sum(1, keepdim=True)).pow(2).mean()
    optimizador.zero_grad(); perdida.backward(); optimizador.step()
    guardar_punto(epoca, modelo, optimizador)
    print(f"Época {epoca} guardada (pérdida {perdida.item():.3f})")

> **Experimenta:** ejecuta la celda de arriba dos veces. La segunda vez **continúa** desde la última época guardada,
> como si la máquina se hubiera apagado entre medio.

## 2. Resultados que se pueden repetir

In [ ]:
import random
import numpy as np

def fijar_semilla(semilla=42):
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)          # también fija la semilla de la GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fijar_semilla(42); a = torch.randn(3)
fijar_semilla(42); b = torch.randn(3)
print("Misma semilla, mismos números:", torch.equal(a, b))

In [ ]:
# Las versiones exactas de lo que usaste: guárdalas junto a tus resultados
!pip freeze | grep -iE "^(torch|transformers|datasets|pandas|numpy|scikit-learn)==" > versiones.txt
!cat versiones.txt

## 3. Errores comunes en cuadernos compartidos (y cómo evitarlos)

| Error | Qué pasa | Cómo evitarlo |
|---|---|---|
| Rutas fijas al Drive de otra persona (`/content/drive/MyDrive/MiCarpeta/...`) | A quien abre el cuaderno le falla: esa carpeta no existe en **su** Drive | Una sola variable `CARPETA` al inicio, que cada uno ajusta, y `Path` para armar rutas |
| Claves en un archivo `.py` subido a Drive o pegadas en una celda | Quien tenga el cuaderno o la carpeta ve la clave | El panel 🔑 Secretos (tema 4) |
| `pip install` sin versión, a mitad del cuaderno | Mañana se instala otra versión, cambian los resultados o hay que reiniciar el entorno | Todas las instalaciones en las primeras celdas, con `==versión` |
| Montar Drive varias veces | Avisos y celdas que dependen del orden | Montarlo **una vez**, al principio |
| Celdas ejecutadas a mano en desorden | El cuaderno «funciona» solo en tu sesión | Antes de compartir: **Entorno de ejecución → Reiniciar y ejecutar todas** |
| Compartir como «Editor» con todos | Cualquiera puede cambiar tu cuaderno | Compartir como **Lector**; cada uno guarda su copia |

## 4. Uso responsable

- Respeta las prohibiciones de Colab: nada de minar criptomonedas, *deepfakes*, descifrar contraseñas ni **varias
  cuentas para saltarse los límites** ([FAQ](https://research.google.com/colaboratory/faq.html)).
- Las funciones de IA de Colab (Gemini) piden **18 años o más** y, según la FAQ, guardan tus mensajes y respuestas
  **hasta 18 meses**: no les pegues datos personales ni secretos.
- Libera la GPU cuando termines (**Entorno de ejecución → Desconectar y borrar el entorno de ejecución**): otros la
  esperan.
- ¿Necesitas más? Colab Pro, Pro+ y Pay As You Go dan más disponibilidad según tus unidades de cómputo; revisa los
  precios vigentes en la página de planes antes de pagar.

## Pruébalo tú

Escribe `ruta_resultado(nombre)`, que devuelva la ruta `CARPETA / "resultados" / nombre` **creando la carpeta
`resultados` si no existe**. Así nadie depende de tus rutas.

In [ ]:
def ruta_resultado(nombre):
    # ← tu código aquí
    return None

In [ ]:
# Comprobación
r = ruta_resultado("prueba.txt")
if r is None:
    print("Aún no lo resuelves.")
elif r == CARPETA / "resultados" / "prueba.txt" and r.parent.is_dir():
    print("✅ ¡Correcto!", r)
else:
    print("❌ Revisa la ruta o que la carpeta exista:", r)

## Resumen

- **Puntos de control en Drive** para retomar tras una desconexión.
- **Semillas y versiones fijas** para repetir resultados.
- Una sola `CARPETA`, claves en Secretos, instalaciones al inicio y **Reiniciar y ejecutar todas** antes de compartir.

**Fuentes:** [FAQ de Colab](https://research.google.com/colaboratory/faq.html) · [PyTorch · reproducibilidad](https://docs.pytorch.org/docs/stable/notes/randomness.html) · [PyTorch · puntos de control](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)